<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task3/Task3_Model2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Task 3 — ML Model Portfolio

#import and start pyspark
!pip install pyspark -q

from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print('Spark Successfully Started!')

Spark Successfully Started!


In [2]:
#load data from task 2
#mount google drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

# Load preprocessed train/test data saved in Task 2
train_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/train_data.parquet'
)
test_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/test_data.parquet'
)

print(f"Training rows: {train_df.count():,}")
print(f"Testing rows: {test_df.count():,}")

# Use a smaller sample for model tuning - due to RAM issue
train_sample = train_df.sample(0.03, seed=42)
test_sample = test_df.sample(0.03, seed=42)

print(f"Training sample rows: {train_sample.count():,}")
print(f"Testing sample rows: {test_sample.count():,}")

print("Data loaded successfully!")

Mounted at /content/drive
Training rows: 1,469,696
Testing rows: 367,299
Training sample rows: 44,296
Testing sample rows: 11,046
Data loaded successfully!


In [5]:
#MODEL 2: Random Forest Regressor-
#predicts exact cost
#Captures non-linear interactions between region, drug, GP practice

from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator #automatically tests all settings and picks the best

rfr_Model = RandomForestRegressor(
    featuresCol='scaled_features',
    labelCol='ACTUAL_COST',
    seed=42
)

print("Random Forest Regressor defined!")

Random Forest Regressor defined!


In [7]:
# grid creation - lightweight for memory
rf_grid = ParamGridBuilder() \
    .addGrid(rfr_Model.numTrees, [20]) \
    .addGrid(rfr_Model.maxDepth, [5]) \
    .build()

# Evaluator for regression models
rf_evaluator = RegressionEvaluator(
    labelCol='ACTUAL_COST',
    predictionCol='prediction',
    metricName='rmse'
)

rf_cv = CrossValidator(
    estimator=rfr_Model,
    estimatorParamMaps=rf_grid,
    evaluator=rf_evaluator,
    numFolds=2,
    seed=42,
    parallelism=1
)

rf_r2_evaluator = RegressionEvaluator(
    labelCol='ACTUAL_COST',
    predictionCol='prediction',
    metricName='r2'
)

rf_mae_evaluator = RegressionEvaluator(
    labelCol='ACTUAL_COST',
    predictionCol='prediction',
    metricName='mae'
)

print("Random Forest CrossValidator configured!")
print(f"numTrees: 20, maxDepth: 5") #reduced for Colab memory

Random Forest CrossValidator configured!
numTrees: 20, maxDepth: 5
